# News Topic Classifier Using BERT


## Problem Statement & Objective

* **Problem Statement:** Online news platforms generate thousands of articles daily across various domains. Manually sorting these articles into specific sections is time-consuming and prone to human error. Automating this process using text classification is essential for efficient content organization and recommendation.

* **Objective:** Fine-tune a pretrained bert-base-uncased Transformer model to automatically classify news headlines into four distinct categories: World, Sports, Business, and Sci/Tech. The target is to build an end-to-end pipeline that achieves high classification performance and is deployable for live interactive testing.

## 1. Environment Setup

In [ ]:
!pip install -q -U transformers datasets scikit-learn torch torchvision

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE} | GPU: {torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'None'}")

## 2. Load & Subset Dataset

**Dataset Used:** The AG News dataset (specifically using the official repository path fancyzhx/ag_news via the Hugging Face datasets library).

**Data Splitting:** To balance computational speed and training quality in the environment, subsetting was applied:

**Training Set:** 5,000 samples (shuffled with a fixed seed of 42).

**Evaluation Set:** 1,000 samples (shuffled with a fixed seed of 42).

Note: Hugging Face API key is required

In [ ]:
# load dataset from Hugging Face
raw = load_dataset('fancyzhx/ag_news')

In [ ]:
TRAIN_SIZE, TEST_SIZE = 5000, 1000
train_ds = raw['train'].shuffle(seed=42).select(range(TRAIN_SIZE))
test_ds  = raw['test'].shuffle(seed=42).select(range(TEST_SIZE))

# features describes the schema of the dataset — like column types in a table.
# The 'label' feature has a .names attribute that gives you the human-readable category names:
# Without this, labels would just be numbers (0, 1, 2, 3)
LABELS = raw['train'].features['label'].names # output: Classes: ['World', 'Sports', 'Business', 'Sci/Tech']
print(f"Classes: {LABELS}")

# prints the first row of our training subset
print(train_ds[0])

## 3. Tokenize

**Tokenization:**
Texts were tokenized using the standard BertTokenizer (bert-base-uncased).\
Sentences were padded and truncated to a maximum length of 128 tokens.\
Input tensors were formatted explicitly for PyTorch (input_ids, attention_mask and label).

In [ ]:
# the bert-base-uncased model uses 'bert-base-uncased' tokenizer
# that converts all of the text to lowercase and tokenize it

""" BERT (Bidirectional Encoder Representations from Transformers) is a powerful
pre-trained language model made by Google.
bert-base-uncased means it's the base size (not too big, not too small)
and uncased (treats Hello and hello as the same word — everything is lowercased)"""
MODEL_CKPT = 'bert-base-uncased'

# AutoTokenizer automatically finds the right tokenizer for the model
tokenizer  = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize(batch):
    """
    batch['text'] Grabs the text column from each row
    truncation=True If text is too long, cut it off at max_length
    padding='max_length'If text is too short, pad it with zeros up to max_length
    max_length=128 Every sequence will be exactly 128 tokens long
    """
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

train_enc = train_ds.map(tokenize, batched=True)
# .map()Applies a function to every row in the dataset
# tokenize Our function from above
# batched=True Process rows in groups instead of one-by-one — much faster
# train_enc Short for train encoded — the tokenized training data
# test_enc Short for test encoded — the tokenized test data
test_enc  = test_ds.map(tokenize, batched=True)


# This tells the dataset to return PyTorch tensors instead of plain Python lists, and only keep three columns:
# input_ids The token IDs - numbers representing each word/subword
# attention_mask tells the model which tokens are real (1) vs padding (0)
# label The category (0=World, 1=Sports, 2=Business, 3=Sci/Tech)
train_enc.set_format('torch', columns=['input_ids','attention_mask','label'])
test_enc.set_format('torch',  columns=['input_ids','attention_mask','label'])

## 4. Load Pretrained BERT

## Model Development & TrainingModel Architecture:
AutoModelForSequenceClassification initialized with pretrained bert-base-uncased weights, adapting the final classification layer to output logits for 4 target classes.

In [ ]:
# # enumerate(LABELS) gives you:
# (0, 'World'), (1, 'Sports'), (2, 'Business'), (3, 'Sci/Tech')

# id2label = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}
# label2id = {'World': 0, 'Sports': 1, 'Business': 2, 'Sci/Tech': 3}

# id2label and label2id are stored as part of the model's configuration.

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(LABELS),
    id2label={i: l for i, l in enumerate(LABELS)},
    label2id={l: i for i, l in enumerate(LABELS)}
).to(DEVICE)

## 5. Training Configuration

## Hardware Acceleration:
Training was executed on a Tesla T4 GPU using mixed-precision (fp16=True) to accelerate computation.\
**Epochs: 5**\
**Batch Size:** 16 for training / 32 for evaluation\
**Learning Rate:** $2 \times 10^{-5}$\
**Weight Decay:** 0.01

**Optimization Strategy:** Best model selection based on the highest validation F1-Score (load_best_model_at_end=True).

In [ ]:
# # Example logits for one news article:
# [2.1,  8.7,  0.3,  1.2]
# World Sports Business Sci/Tech
# Higher score = model is more confident about that category.


# preds = np.argmax(logits, axis=-1)
# argmax finds the index of the highest value — that index is our predicted label:


# Accuracy = Correct Predictions / Total Predictions
# F1 = balance between catching all correct cases
#      and not making too many wrong guesses


9 correct out of 10 → 90% accuracy
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1':       f1_score(labels, preds, average='weighted')
    }

args = TrainingArguments(
    output_dir='./bert-ag-news', # Where to save the model checkpoints
    num_train_epochs=5, # Pass through the entire training data 5 times
    per_device_train_batch_size=16, # Train on 16 samples at a time
    per_device_eval_batch_size=32, # Evaluate on 32 samples at a time
    learning_rate=2e-5, # 2e-5 is scientific notation for 0.00002 - How big each weight update step is
    weight_decay=0.01, # Regularization — prevents overfitting
    eval_strategy='epoch', # Evaluate the model after every epoch
    save_strategy='epoch', # Save a checkpoint after every epoch
    load_best_model_at_end=True, # At the end, keep the best model not the last
    metric_for_best_model='f1', # Define "best" by F1 score
    logging_steps=50, # Print training logs every 50 steps
    fp16=(DEVICE=='cuda'), # True if GPU - Use 16-bit precision on GPU for speed
    report_to='none' # Don't send logs to wandb/tensorboard
)

## 6. Fine-Tune

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_enc,
    eval_dataset=test_enc,
    compute_metrics=compute_metrics
)

trainer.train()
""""
- trainer.train() does an enormous amount of work under the hood
For each epoch (5 total):
    For each batch of 16 samples:

        1. FORWARD PASS
           Feed batch into BERT → get logits (raw scores)

        2. LOSS CALCULATION
           Compare logits to true labels → compute how wrong we are

        3. BACKWARD PASS
           Calculate gradients — figure out which weights caused the error

        4. WEIGHT UPDATE
           Nudge weights slightly in the right direction (learning rate = 2e-5)

    End of epoch → evaluate on test set → compute accuracy & F1
    Save checkpoint if best F1 so far
"""

## 7. Evaluate

## Evaluation with Relevant Metrics
The model was evaluated on the unseen validation dataset using **Accuracy** and a Weighted **F1-Score** (to account for any slight label variations).

In [ ]:
# For each batch of 32 samples (eval batch size):
    # 1. Feed into model → get logits
    # 2. Convert logits → predictions (argmax)
    # 3. Compare predictions vs true labels
    # 4. Compute accuracy & F1 via compute_metrics()
# return all results as a dictionary

results = trainer.evaluate()

print(f"Accuracy : {results['eval_accuracy']:.4f}")
print(f"F1 Score : {results['eval_f1']:.4f}")

## 8. Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


"""
trainer.predict(test_enc) Runs the model on test data, returns raw logits
np.argmax(..., axis=-1) Converts logits → predicted label numbers
preds_out.label_ids Grabs the true labels to compare against
"""
preds_out = trainer.predict(test_enc)
y_pred = np.argmax(preds_out.predictions, axis=-1)
y_true = preds_out.label_ids


# This produces a grid comparing true labels vs predicted labels.
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=LABELS) # Wraps the matrix with label names instead of numbers
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
plt.title('Confusion Matrix — BERT on AG News')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## Visualizations & Deployment
**Confusion Matrix:** A confusion matrix was plotted using matplotlib and scikit-learn to review exact distribution patterns and cross-class classification errors.

**Insights:** The model shows incredibly high precision across all classes. Minor misclassifications occur strictly between highly adjacent contexts (such as certain "Business" news getting flagged as "Sci/Tech" or "World").

**Live Deployment:** The final model and tokenizer weights were locally compiled under ./bert-ag-news-final and successfully deployed to a web UI using Gradio. This deployment features real-time classification input fields alongside example test vectors.

Also deployed on Hugging Face Space : [news-topic-classifier-BERT](https://huggingface.co/spaces/johnasad09/news-topic-classifier-bert)

## 9. Inference on Custom Headlines

In [ ]:
from transformers import pipeline

clf = pipeline('text-classification', model=model, tokenizer=tokenizer, device=0 if DEVICE=='cuda' else -1)

headlines = [
    "NASA launches new Mars rover mission",
    "Stock market hits record high amid Fed signals",
    "Real Madrid wins Champions League final",
    "UN Security Council meets over Gaza ceasefire"
]

for h in headlines:
    out = clf(h)[0]
    print(f"{out['label']:12s} ({out['score']:.2%})  →  {h}")

## 10. Save Model

In [ ]:
model.save_pretrained('./bert-ag-news-final')
tokenizer.save_pretrained('./bert-ag-news-final')
print('Model saved to ./bert-ag-news-final')

# Deploying the model with Gradio

In [ ]:
# Install Gradio
!pip install -q gradio

import gradio as gr
from transformers import pipeline

# 1. Create the classification pipeline using your trained model and tokenizer
# (This uses the GPU if available, matching your current notebook setup)
clf = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    device=0 if DEVICE=='cuda' else -1
)

# 2. Define the prediction function that Gradio will call
def predict_news_topic(headline):
    if not headline.strip():
        return "Please enter a valid headline."

    # Get prediction from the pipeline
    output = clf(headline)[0]
    label = output['label']
    confidence = output['score']

    # Return a formatted string with the result
    return f"Prediction: {label} ({confidence:.2%} confidence)"

# 3. Build the Gradio Interface
demo = gr.Interface(
    fn=predict_news_topic,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Enter a news headline here...",
        label="News Headline"
    ),
    outputs=gr.Textbox(label="Model Prediction"),
    title="📰 News Topic Classifier (BERT)",
    description="Enter a news headline to classify it into one of the 4 AG News categories: World, Sports, Business, or Sci/Tech.",
    examples=[
        ["NASA's James Webb telescope captures stunning new image of a distant galaxy."],
        ["Global markets slide as inflation concerns grip investors."],
        ["Formula 1 championship race tightens after dramatic weekend finish."],
        ["Diplomats gather at the UN to discuss the ongoing humanitarian crisis."]
    ]
)

# 4. Launch the interface with a public link
demo.launch(share=True)

In [ ]:
# Run this in a new cell in Colab to zip your final model folder
!zip -r bert_model.zip ./bert-ag-news-final

## Final Summary & Performance Insights
### Training & Convergence Summary
**Validation Performance Peak:**\
The model reached its optimal balance of performance and generalization at Epoch 4, achieving a peak Validation Accuracy of 91.10% and a Weighted F1-Score of 91.09%.

**Generalization Power:**\
The model demonstrates robust generalization capabilities. Even when trained on a drastically constrained subset of the AG News data (only 5,000 samples out of the original 120,000 training pool), the pre-trained contextual representations of BERT allowed it to adapt to the downstream classification task with high efficiency.

**Loss Trajectory & Early Stopping:**\
The training loss steadily decreased from 0.3541 (Epoch 1) down to 0.0503 (Epoch 5). However, validation loss reached its minimum at Epoch 2 (0.3116) and began slightly tracking upwards afterward. Because load_best_model_at_end=True was configured against the F1-metric, the trainer successfully bypassed the overfitted Epoch 5 checkpoints and restored the superior weights from Epoch 4.

### Error Analysis (Confusion Matrix Insights)
Looking closely at the generated Confusion Matrix, the model exhibits definitive strengths and clear contextual vulnerabilities across the four news domains:

**The Strongest Class (Sports):**\
The model performs exceptionally well on Sports news (correctly predicting 240 samples). Out of the 1,000 evaluation rows, it only misclassified 6 total sports headlines. This indicates that sports jargon (e.g., team names, tournaments, verbs like "wins", "defeats") provides highly distinct textual boundaries that BERT isolates effortlessly.

**Contextual Overlap (Business vs. Sci/Tech):**\
The highest rate of error occurs between Business and Sci/Tech.\
Specifically, 25 Business headlines were misclassified as Sci/Tech.

**Insight:**\
This pattern occurs because corporate news frequently covers the tech sector (e.g., quarterly earnings of Apple, tech mergers, or semiconductor stock changes). Vocabulary like "shares", "market values", or specific tech company names spans both fields, making it the most prominent point of ambiguity for the encoder.

Geopolitical Noise (World vs. Sci/Tech & Business):\
*15 World headlines were misclassified as Sci/Tech.*\
**Insight:**\
This commonly stems from global infrastructure news, cyber warfare, defense technology development, or international space missions (e.g., "NASA launches new Mars rover") where global politics and technological advancements are heavily intertwined.

### Production Deployment Viability
**Inference Confidences:**\
During real-time inference testing, the model yielded confidence scores exceeding 99% on straightforward headlines (such as "NASA launches new Mars rover mission" == Sci/Tech 99.60%).\
**Edge Case Handling:**\
When exposed to complex, cross-domain user queries via the live Gradio interface (e.g., "Govt revises business closing hours nationwide"), the model correctly mapped the intent to Business with a healthy confidence of 96.58%. This proves that the fine-tuned BERT architecture is relying on deep structural context rather than just keyword matching, rendering it highly viable for an automated, production-level news aggregation system.